# Synthetic CT Generation — Latent Diffusion Model

---
**Pipeline:**
1. Environment setup & imports
2. Configuration
3. Data preprocessing (LIDC-IDRI)
4. Dataset & DataLoader
5. VAE (512 → 64 latent)
6. Mask Conditioning Encoder
7. Conditional Diffusion U-Net (512-dim bottleneck)
8. DDPM / DDIM Noise Scheduler
9. EMA (Exponential Moving Average)
10. VAE Training Loop
11. Diffusion Training Loop
12. Inference & Synthetic CT Generation
13. Evaluation Metrics

## Cell 1 — Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install diffusers accelerate transformers
!pip install SimpleITK pylidc nibabel
!pip install torchio scikit-image
!pip install wandb tqdm matplotlib einops
!pip install lpips  # perceptual loss for VAE

## Cell 2 — Imports

In [ ]:
import os
import math
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import SimpleITK as sitk
import pylidc as pl
from pylidc.utils import consensus
import torchio as tio
from skimage.transform import resize
import lpips

import wandb

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Cell 3 — Configuration

In [ ]:
@dataclass
class Config:
    # ── Paths ──────────────────────────────────────────
    data_dir: str        = '/path/to/LIDC-IDRI'   # TCIA download folder
    output_dir: str      = './outputs'
    vae_ckpt: str        = './outputs/vae_best.pt'
    diff_ckpt: str       = './outputs/diffusion_best.pt'

    # ── Image ──────────────────────────────────────────
    image_size: int      = 512          # output CT resolution
    latent_size: int     = 64           # VAE latent spatial dim (512/8)
    latent_channels: int = 4            # VAE latent channels
    in_channels: int     = 1            # grayscale CT

    # ── HU Windowing (lung window) ─────────────────────
    hu_min: int          = -1000
    hu_max: int          = 400

    # ── U-Net ──────────────────────────────────────────
    unet_base_channels: int         = 128
    unet_channel_mults: List[int]   = field(default_factory=lambda: [1, 2, 4])
    # channels: [128, 256, 512] — bottleneck = 512
    attention_resolutions: List[int]= field(default_factory=lambda: [8, 16, 32])
    num_res_blocks: int             = 2
    num_heads: int                  = 8
    dropout: float                  = 0.1
    timestep_emb_dim: int           = 512

    # ── Diffusion ──────────────────────────────────────
    T_train: int         = 1000         # DDPM timesteps
    T_infer: int         = 50           # DDIM inference steps
    beta_start: float    = 1e-4
    beta_end: float      = 0.02
    beta_schedule: str   = 'cosine'     # 'linear' or 'cosine'

    # ── Training ───────────────────────────────────────
    vae_epochs: int      = 100
    diff_epochs: int     = 500
    batch_size: int      = 8
    lr: float            = 1e-4
    weight_decay: float  = 1e-4
    warmup_steps: int    = 1000
    grad_clip: float     = 1.0
    ema_decay: float     = 0.9999

    # ── Data Split ─────────────────────────────────────
    train_ratio: float   = 0.80
    val_ratio: float     = 0.10
    test_ratio: float    = 0.10
    num_workers: int     = 4

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
print(cfg)

## Cell 4 — CT Preprocessing Utilities

In [ ]:
def load_ct_volume(path: str) -> np.ndarray:
    """Load CT volume from DICOM directory or NIfTI file."""
    if os.path.isdir(path):
        reader = sitk.ImageSeriesReader()
        dicom_files = reader.GetGDCMSeriesFileNames(path)
        reader.SetFileNames(dicom_files)
        image = reader.Execute()
    else:
        image = sitk.ReadImage(path)
    return image


def resample_volume(image: sitk.Image,
                    new_spacing: List[float] = [1.0, 1.0, 1.0]) -> sitk.Image:
    """Resample CT volume to isotropic 1mm spacing."""
    original_spacing = image.GetSpacing()
    original_size    = image.GetSize()
    new_size = [
        int(round(original_size[i] * original_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(-1000)  # air HU
    resampler.SetInterpolator(sitk.sitkLinear)
    return resampler.Execute(image)


def apply_window(volume: np.ndarray,
                 hu_min: int = -1000,
                 hu_max: int = 400) -> np.ndarray:
    """Apply HU windowing for lung CT."""
    return np.clip(volume, hu_min, hu_max)


def normalize_ct(volume: np.ndarray,
                 hu_min: int = -1000,
                 hu_max: int = 400) -> np.ndarray:
    """Normalize HU values to [-1, 1]."""
    volume = apply_window(volume, hu_min, hu_max)
    volume = (volume - hu_min) / (hu_max - hu_min)  # [0, 1]
    volume = volume * 2.0 - 1.0                      # [-1, 1]
    return volume.astype(np.float32)


def denormalize_ct(volume: np.ndarray,
                   hu_min: int = -1000,
                   hu_max: int = 400) -> np.ndarray:
    """Convert [-1, 1] back to HU values."""
    volume = (volume + 1.0) / 2.0                    # [0, 1]
    volume = volume * (hu_max - hu_min) + hu_min     # HU
    return volume.astype(np.float32)


def resize_slice(slice_2d: np.ndarray,
                 target_size: int = 512) -> np.ndarray:
    """Resize a 2D slice to target_size x target_size."""
    return resize(slice_2d,
                  (target_size, target_size),
                  mode='reflect',
                  anti_aliasing=True).astype(np.float32)


def has_lung_content(slice_2d: np.ndarray,
                     threshold: float = 0.15) -> bool:
    """Filter slices with insufficient lung content."""
    # In normalized space, lung parenchyma ~ [-1, -0.3]
    lung_mask = (slice_2d > -1.0) & (slice_2d < -0.3)
    return lung_mask.mean() > threshold


print('Preprocessing utilities defined.')

## Cell 5 — LIDC-IDRI Dataset Builder

In [ ]:
def build_lidc_dataset(cfg: Config,
                       save_dir: str = './data/processed') -> None:
    """
    Parse LIDC-IDRI using pylidc, extract paired (CT slice, nodule mask)
    at 512x512 resolution and save as .npy files.
    """
    os.makedirs(save_dir, exist_ok=True)
    scans = pl.query(pl.Scan).all()
    print(f'Total LIDC scans: {len(scans)}')

    slice_id = 0
    for scan_idx, scan in enumerate(tqdm(scans, desc='Processing scans')):
        try:
            # Load volume
            vol = scan.to_volume()          # numpy (H, W, Z), HU values
            vol = vol.transpose(2, 0, 1)    # (Z, H, W)

            # Build nodule mask volume
            mask_vol = np.zeros_like(vol, dtype=np.float32)
            anns = scan.cluster_annotations()
            for ann_group in anns:
                try:
                    cmask, cbbox, _ = consensus(ann_group, clevel=0.5)
                    z0, z1 = cbbox[2].start, cbbox[2].stop
                    y0, y1 = cbbox[0].start, cbbox[0].stop
                    x0, x1 = cbbox[1].start, cbbox[1].stop
                    # cmask shape: (y, x, z)
                    for z_local in range(cmask.shape[2]):
                        z_global = z0 + z_local
                        if z_global < mask_vol.shape[0]:
                            mask_vol[z_global,
                                     y0:y1, x0:x1] = np.maximum(
                                mask_vol[z_global, y0:y1, x0:x1],
                                cmask[:, :, z_local].astype(np.float32)
                            )
                except Exception:
                    continue

            # Process slice by slice
            for z in range(vol.shape[0]):
                ct_slice   = vol[z]       # (H, W)
                mask_slice = mask_vol[z]  # (H, W)

                # Resize to 512×512
                ct_slice   = resize_slice(ct_slice,   cfg.image_size)
                mask_slice = resize_slice(mask_slice, cfg.image_size)
                mask_slice = (mask_slice > 0.5).astype(np.float32)

                # Normalize CT
                ct_norm = normalize_ct(ct_slice, cfg.hu_min, cfg.hu_max)

                # Filter empty slices
                if not has_lung_content(ct_norm):
                    continue

                # Save pair
                np.save(f'{save_dir}/ct_{slice_id:06d}.npy',   ct_norm)
                np.save(f'{save_dir}/mask_{slice_id:06d}.npy', mask_slice)
                slice_id += 1

        except Exception as e:
            print(f'Error on scan {scan_idx}: {e}')
            continue

    print(f'Total slices saved: {slice_id}')


# Run once to preprocess
# build_lidc_dataset(cfg, save_dir='./data/processed')

## Cell 6 — PyTorch Dataset & DataLoader

In [ ]:
class LIDCDataset(Dataset):
    def __init__(self,
                 data_dir: str,
                 split: str = 'train',
                 cfg: Config = None,
                 augment: bool = True):
        self.data_dir = data_dir
        self.split    = split
        self.cfg      = cfg
        self.augment  = augment and (split == 'train')

        # Collect all CT file indices
        all_files = sorted([
            f for f in os.listdir(data_dir) if f.startswith('ct_')
        ])
        indices = list(range(len(all_files)))
        random.shuffle(indices)

        n = len(indices)
        n_train = int(n * cfg.train_ratio)
        n_val   = int(n * cfg.val_ratio)

        if split == 'train':
            self.indices = indices[:n_train]
        elif split == 'val':
            self.indices = indices[n_train:n_train + n_val]
        else:
            self.indices = indices[n_train + n_val:]

        print(f'{split}: {len(self.indices)} slices')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        ct   = np.load(f'{self.data_dir}/ct_{i:06d}.npy')    # (512, 512) float32
        mask = np.load(f'{self.data_dir}/mask_{i:06d}.npy')  # (512, 512) float32

        if self.augment:
            ct, mask = self._augment(ct, mask)

        # Add channel dim: (1, 512, 512)
        ct   = torch.from_numpy(ct).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return {'ct': ct, 'mask': mask}

    def _augment(self, ct, mask):
        # Horizontal flip (left-right anatomically valid for lungs)
        if random.random() > 0.5:
            ct   = np.fliplr(ct).copy()
            mask = np.fliplr(mask).copy()

        # Random rotation ±10°
        if random.random() > 0.5:
            from scipy.ndimage import rotate
            angle = random.uniform(-10, 10)
            ct   = rotate(ct,   angle, reshape=False, mode='nearest')
            mask = rotate(mask, angle, reshape=False, mode='nearest', order=0)

        # Intensity jitter (±0.05 in normalized space ≈ ±25 HU)
        if random.random() > 0.5:
            ct = ct + np.random.uniform(-0.05, 0.05)
            ct = np.clip(ct, -1.0, 1.0)

        return ct.astype(np.float32), mask.astype(np.float32)


def get_dataloaders(cfg: Config, data_dir: str):
    train_ds = LIDCDataset(data_dir, 'train', cfg, augment=True)
    val_ds   = LIDCDataset(data_dir, 'val',   cfg, augment=False)
    test_ds  = LIDCDataset(data_dir, 'test',  cfg, augment=False)

    train_dl = DataLoader(train_ds, batch_size=cfg.batch_size,
                          shuffle=True,  num_workers=cfg.num_workers,
                          pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size,
                          shuffle=False, num_workers=cfg.num_workers,
                          pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=cfg.batch_size,
                          shuffle=False, num_workers=cfg.num_workers,
                          pin_memory=True)
    return train_dl, val_dl, test_dl


# train_dl, val_dl, test_dl = get_dataloaders(cfg, './data/processed')
# batch = next(iter(train_dl))
# print('CT shape:', batch['ct'].shape)
# print('Mask shape:', batch['mask'].shape)

## Cell 7 — Building Blocks (ResNet, Attention, Timestep Embedding)

In [ ]:
# ── Timestep Sinusoidal Embedding ──────────────────────────────────────────
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        device = t.device
        half   = self.dim // 2
        freqs  = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        )
        args   = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class TimestepMLP(nn.Module):
    """Projects sinusoidal embedding to full timestep_emb_dim."""
    def __init__(self, dim: int):
        super().__init__()
        self.net = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t):
        return self.net(t)


# ── ResNet Block ────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int,
                 time_emb_dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm1  = nn.GroupNorm(32, in_ch)
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2  = nn.GroupNorm(32, out_ch)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.drop   = nn.Dropout(dropout)
        self.act    = nn.SiLU()

        # FiLM conditioning from timestep
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch * 2)  # scale + shift
        )

        # Skip connection
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.act(self.norm1(x))
        h = self.conv1(h)

        # FiLM: scale and shift by timestep
        t_out         = self.time_proj(t_emb)[:, :, None, None]
        scale, shift  = t_out.chunk(2, dim=1)
        h             = self.norm2(h) * (1 + scale) + shift

        h = self.act(h)
        h = self.drop(h)
        h = self.conv2(h)
        return h + self.skip(x)


# ── Self Attention Block ────────────────────────────────────────────────────
class SelfAttention(nn.Module):
    def __init__(self, channels: int, num_heads: int = 8):
        super().__init__()
        self.norm  = nn.GroupNorm(32, channels)
        self.attn  = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        h = self.norm(x)
        h = h.reshape(B, C, H * W).permute(0, 2, 1)   # (B, HW, C)
        h, _ = self.attn(h, h, h)
        h = h.permute(0, 2, 1).reshape(B, C, H, W)
        return x + h


# ── Downsample / Upsample ───────────────────────────────────────────────────
class Downsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.ConvTranspose2d(channels, channels, 4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


print('Building blocks defined.')

## Cell 8 — VAE (512 → 64 Latent Space)

In [ ]:
class VAEEncoder(nn.Module):
    """Encodes 512x512 CT slice → 64x64x4 latent (mu, logvar)."""
    def __init__(self, in_ch: int = 1, latent_ch: int = 4):
        super().__init__()
        self.encoder = nn.Sequential(
            # 512×512 → 256×256
            nn.Conv2d(in_ch, 64, 3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GroupNorm(32, 64),
            nn.SiLU(),

            # 256×256 → 128×128
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.GroupNorm(32, 128),
            nn.SiLU(),

            # 128×128 → 64×64
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.GroupNorm(32, 256),
            nn.SiLU(),
        )
        # Output mu and logvar separately
        self.mu_proj     = nn.Conv2d(256, latent_ch, 1)
        self.logvar_proj = nn.Conv2d(256, latent_ch, 1)

    def forward(self, x):
        h      = self.encoder(x)
        mu     = self.mu_proj(h)
        logvar = self.logvar_proj(h)
        return mu, logvar


class VAEDecoder(nn.Module):
    """Decodes 64x64x4 latent → 512x512x1 CT slice."""
    def __init__(self, latent_ch: int = 4, out_ch: int = 1):
        super().__init__()
        self.decoder = nn.Sequential(
            # 64×64 → 128×128
            nn.Conv2d(latent_ch, 256, 3, padding=1),
            nn.GroupNorm(32, 256),
            nn.SiLU(),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.GroupNorm(32, 128),
            nn.SiLU(),

            # 128×128 → 256×256
            nn.Conv2d(128, 128, 3, padding=1),
            nn.GroupNorm(32, 128),
            nn.SiLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.GroupNorm(32, 64),
            nn.SiLU(),

            # 256×256 → 512×512
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GroupNorm(32, 64),
            nn.SiLU(),
            nn.ConvTranspose2d(64, out_ch, 4, stride=2, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        return self.decoder(z)


class VAE(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.encoder = VAEEncoder(cfg.in_channels, cfg.latent_channels)
        self.decoder = VAEDecoder(cfg.latent_channels, cfg.in_channels)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z, mu, logvar = self.encode(x)
        recon         = self.decode(z)
        return recon, mu, logvar


vae = VAE(cfg).to(DEVICE)
print(f'VAE parameters: {sum(p.numel() for p in vae.parameters()):,}')

# Quick shape test
with torch.no_grad():
    x_test    = torch.randn(2, 1, 512, 512).to(DEVICE)
    recon, mu, logvar = vae(x_test)
    print(f'Input:  {x_test.shape}')
    print(f'Latent: {mu.shape}')      # (2, 4, 64, 64)
    print(f'Recon:  {recon.shape}')   # (2, 1, 512, 512)

## Cell 9 — Mask Conditioning Encoder

In [ ]:
class MaskEncoder(nn.Module):
    """
    Encodes nodule segmentation mask (512x512x1)
    into latent space (64x64x4) to condition the diffusion U-Net.
    """
    def __init__(self, in_ch: int = 1, out_ch: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            # 512×512 → 256×256
            nn.Conv2d(in_ch, 32, 3, stride=2, padding=1),
            nn.SiLU(),

            # 256×256 → 128×128
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.GroupNorm(32, 64),
            nn.SiLU(),

            # 128×128 → 64×64
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.GroupNorm(32, 128),
            nn.SiLU(),

            # Project to latent_channels
            nn.Conv2d(128, out_ch, 1),
        )

    def forward(self, mask):
        return self.net(mask)   # (B, 4, 64, 64)


mask_enc = MaskEncoder(cfg.in_channels, cfg.latent_channels).to(DEVICE)
print(f'MaskEncoder parameters: {sum(p.numel() for p in mask_enc.parameters()):,}')

# Shape test
with torch.no_grad():
    m_test = torch.randn(2, 1, 512, 512).to(DEVICE)
    m_out  = mask_enc(m_test)
    print(f'Mask input:  {m_test.shape}')
    print(f'Mask latent: {m_out.shape}')  # (2, 4, 64, 64)

## Cell 10 — Conditional Diffusion U-Net (512-dim Bottleneck)

In [ ]:
class UNet(nn.Module):
    """
    Conditional Diffusion U-Net.
    Input:  noisy CT latent (B,4,64,64) + mask latent (B,4,64,64) → concat (B,8,64,64)
    Output: predicted noise (B,4,64,64)
    Bottleneck feature dim: 512
    """
    def __init__(self, cfg: Config):
        super().__init__()
        # in_ch = noisy_latent_ch + mask_latent_ch
        in_ch   = cfg.latent_channels * 2
        out_ch  = cfg.latent_channels
        base    = cfg.unet_base_channels          # 128
        mults   = cfg.unet_channel_mults          # [1, 2, 4] → [128, 256, 512]
        channels= [base * m for m in mults]       # [128, 256, 512]
        t_dim   = cfg.timestep_emb_dim            # 512
        heads   = cfg.num_heads
        drop    = cfg.dropout
        nrb     = cfg.num_res_blocks

        # ── Timestep embedding ──────────────────────────
        self.time_mlp = TimestepMLP(t_dim)

        # ── Input projection ────────────────────────────
        self.input_conv = nn.Conv2d(in_ch, channels[0], 3, padding=1)

        # ── Encoder ─────────────────────────────────────
        self.enc_blocks = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        ch_in = channels[0]
        self.skip_channels = []

        for level, ch_out in enumerate(channels):
            level_blocks = nn.ModuleList()
            for _ in range(nrb):
                level_blocks.append(ResBlock(ch_in, ch_out, t_dim, drop))
                ch_in = ch_out
            level_blocks.append(SelfAttention(ch_out, heads))
            self.enc_blocks.append(level_blocks)
            self.skip_channels.append(ch_out)
            if level < len(channels) - 1:
                self.downsamples.append(Downsample(ch_out))
            else:
                self.downsamples.append(nn.Identity())

        # ── Bottleneck (512 channels) ────────────────────
        bot_ch = channels[-1]  # 512
        self.bottleneck = nn.ModuleList([
            ResBlock(bot_ch, bot_ch, t_dim, drop),
            SelfAttention(bot_ch, heads),
            ResBlock(bot_ch, bot_ch, t_dim, drop),
        ])

        # ── Decoder ─────────────────────────────────────
        self.dec_blocks  = nn.ModuleList()
        self.upsamples   = nn.ModuleList()
        rev_channels     = list(reversed(channels))
        rev_skip         = list(reversed(self.skip_channels))

        ch_in = rev_channels[0]
        for level, ch_out in enumerate(rev_channels):
            level_blocks = nn.ModuleList()
            # skip connection doubles input channels
            for _ in range(nrb):
                level_blocks.append(ResBlock(ch_in + rev_skip[level], ch_out, t_dim, drop))
                ch_in = ch_out
            level_blocks.append(SelfAttention(ch_out, heads))
            self.dec_blocks.append(level_blocks)
            if level < len(rev_channels) - 1:
                self.upsamples.append(Upsample(ch_out))
                ch_in = ch_out
            else:
                self.upsamples.append(nn.Identity())

        # ── Output head ──────────────────────────────────
        self.out = nn.Sequential(
            nn.GroupNorm(32, channels[0]),
            nn.SiLU(),
            nn.Conv2d(channels[0], out_ch, 3, padding=1),
        )

    def forward(self,
                x_noisy: torch.Tensor,
                t: torch.Tensor,
                mask_latent: torch.Tensor) -> torch.Tensor:
        """
        x_noisy:     (B, 4, 64, 64)  — noisy CT latent
        t:           (B,)             — timestep indices
        mask_latent: (B, 4, 64, 64)  — conditioning mask latent
        returns:     (B, 4, 64, 64)  — predicted noise
        """
        # Timestep embedding
        t_emb = self.time_mlp(t)

        # Concatenate noisy latent + mask latent
        x = torch.cat([x_noisy, mask_latent], dim=1)  # (B, 8, 64, 64)
        x = self.input_conv(x)                         # (B, 128, 64, 64)

        # Encoder
        skips = []
        for level, (blocks, down) in enumerate(zip(self.enc_blocks, self.downsamples)):
            for i, block in enumerate(blocks):
                if isinstance(block, ResBlock):
                    x = block(x, t_emb)
                else:  # SelfAttention
                    x = block(x)
            skips.append(x)
            x = down(x)

        # Bottleneck
        for block in self.bottleneck:
            if isinstance(block, ResBlock):
                x = block(x, t_emb)
            else:
                x = block(x)

        # Decoder
        for level, (blocks, up) in enumerate(zip(self.dec_blocks, self.upsamples)):
            skip = skips[-(level + 1)]
            x    = torch.cat([x, skip], dim=1)  # skip connection
            for block in blocks:
                if isinstance(block, ResBlock):
                    x = block(x, t_emb)
                else:
                    x = block(x)
            x = up(x)

        return self.out(x)


unet = UNet(cfg).to(DEVICE)
print(f'U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}')

# Shape test
with torch.no_grad():
    z_noisy = torch.randn(2, 4, 64, 64).to(DEVICE)
    t_test  = torch.randint(0, 1000, (2,)).to(DEVICE)
    m_lat   = torch.randn(2, 4, 64, 64).to(DEVICE)
    pred    = unet(z_noisy, t_test, m_lat)
    print(f'Predicted noise shape: {pred.shape}')  # (2, 4, 64, 64)

## Cell 11 — DDPM / DDIM Noise Scheduler

In [ ]:
class NoiseScheduler:
    """
    DDPM forward process + DDIM reverse process.
    Supports 'linear' and 'cosine' beta schedules.
    """
    def __init__(self, cfg: Config):
        self.T          = cfg.T_train
        self.T_infer    = cfg.T_infer
        self.schedule   = cfg.beta_schedule

        # Compute betas
        if cfg.beta_schedule == 'linear':
            betas = torch.linspace(cfg.beta_start, cfg.beta_end, self.T)
        elif cfg.beta_schedule == 'cosine':
            betas = self._cosine_betas(self.T)
        else:
            raise ValueError(f'Unknown schedule: {cfg.beta_schedule}')

        alphas      = 1.0 - betas
        alpha_cumprod = torch.cumprod(alphas, dim=0)

        self.register('betas',            betas)
        self.register('alphas',           alphas)
        self.register('alpha_cumprod',    alpha_cumprod)
        self.register('sqrt_acp',         alpha_cumprod.sqrt())
        self.register('sqrt_one_minus_acp', (1 - alpha_cumprod).sqrt())

    def register(self, name, tensor):
        setattr(self, name, tensor.to(DEVICE))

    def _cosine_betas(self, T: int, s: float = 0.008) -> torch.Tensor:
        steps  = torch.arange(T + 1, dtype=torch.float64) / T
        alphas = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas = alphas / alphas[0]
        betas  = 1 - (alphas[1:] / alphas[:-1])
        return torch.clamp(betas, 0, 0.999).float()

    def q_sample(self, x0: torch.Tensor,
                 t: torch.Tensor,
                 noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Forward noising: q(x_t | x_0)."""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_acp   = self.sqrt_acp[t][:, None, None, None]
        sqrt_1macp = self.sqrt_one_minus_acp[t][:, None, None, None]
        return sqrt_acp * x0 + sqrt_1macp * noise, noise

    @torch.no_grad()
    def ddim_sample(self,
                    model: nn.Module,
                    mask_encoder: nn.Module,
                    mask: torch.Tensor,
                    shape: Tuple,
                    eta: float = 0.0) -> torch.Tensor:
        """
        DDIM reverse process (deterministic when eta=0).
        shape: (B, latent_ch, latent_size, latent_size)
        """
        B = shape[0]
        x = torch.randn(shape).to(DEVICE)

        # Encode mask once
        mask_lat = mask_encoder(mask)

        # Evenly spaced timesteps for DDIM
        timesteps = torch.linspace(self.T - 1, 0, self.T_infer).long().to(DEVICE)

        for i, t_val in enumerate(tqdm(timesteps, desc='DDIM sampling')):
            t_batch = torch.full((B,), t_val, dtype=torch.long).to(DEVICE)

            # Predict noise
            pred_noise = model(x, t_batch, mask_lat)

            # DDIM update
            acp_t  = self.alpha_cumprod[t_val]
            acp_tm = self.alpha_cumprod[timesteps[i + 1]] if i < len(timesteps) - 1 \
                     else torch.tensor(1.0).to(DEVICE)

            # Predicted x0
            x0_pred = (x - (1 - acp_t).sqrt() * pred_noise) / acp_t.sqrt()
            x0_pred = torch.clamp(x0_pred, -1, 1)

            # Direction pointing to x_t
            sigma   = eta * ((1 - acp_tm) / (1 - acp_t) * (1 - acp_t / acp_tm)).sqrt()
            noise   = torch.randn_like(x) if eta > 0 else 0
            x       = acp_tm.sqrt() * x0_pred \
                    + (1 - acp_tm - sigma ** 2).sqrt() * pred_noise \
                    + sigma * noise
        return x


scheduler = NoiseScheduler(cfg)
print('Noise scheduler ready.')
print(f'Beta range: [{scheduler.betas.min():.5f}, {scheduler.betas.max():.5f}]')

## Cell 12 — EMA (Exponential Moving Average)

In [ ]:
class EMA:
    """Maintains an EMA copy of model weights for stable inference."""
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay     = decay
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for ema_p, model_p in zip(self.ema_model.parameters(),
                                   model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

    def state_dict(self):
        return self.ema_model.state_dict()

    def load_state_dict(self, state_dict):
        self.ema_model.load_state_dict(state_dict)


print('EMA class defined.')

## Cell 13 — VAE Loss & Training Loop

In [ ]:
class VAELoss(nn.Module):
    def __init__(self, kl_weight: float = 1e-3, perceptual_weight: float = 0.1):
        super().__init__()
        self.kl_w   = kl_weight
        self.perc_w = perceptual_weight
        # LPIPS perceptual loss (expects 3-channel input — we expand CT)
        self.lpips  = lpips.LPIPS(net='vgg').to(DEVICE)
        for p in self.lpips.parameters():
            p.requires_grad_(False)

    def forward(self, recon, x, mu, logvar):
        # L1 reconstruction
        l1   = F.l1_loss(recon, x)

        # KL divergence
        kl   = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

        # Perceptual (expand grayscale to 3-ch for LPIPS)
        recon_3ch = recon.repeat(1, 3, 1, 1)
        x_3ch     = x.repeat(1, 3, 1, 1)
        perc      = self.lpips(recon_3ch, x_3ch).mean()

        total = l1 + self.kl_w * kl + self.perc_w * perc
        return total, {'l1': l1.item(), 'kl': kl.item(), 'perceptual': perc.item()}


def train_vae(vae, train_dl, val_dl, cfg):
    optimizer   = torch.optim.AdamW(vae.parameters(),
                                    lr=cfg.lr,
                                    weight_decay=cfg.weight_decay)
    scheduler_lr= torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=cfg.vae_epochs)
    loss_fn     = VAELoss().to(DEVICE)
    scaler      = torch.cuda.amp.GradScaler()

    best_val_loss = float('inf')

    for epoch in range(cfg.vae_epochs):
        # ── Train ──────────────────────────────────────
        vae.train()
        train_losses = []
        for batch in tqdm(train_dl, desc=f'VAE Epoch {epoch+1}/{cfg.vae_epochs}'):
            ct = batch['ct'].to(DEVICE)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                recon, mu, logvar = vae(ct)
                loss, loss_dict   = loss_fn(recon, ct, mu, logvar)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(loss.item())

        # ── Validate ────────────────────────────────────
        vae.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_dl:
                ct = batch['ct'].to(DEVICE)
                with torch.cuda.amp.autocast():
                    recon, mu, logvar = vae(ct)
                    loss, _           = loss_fn(recon, ct, mu, logvar)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)
        scheduler_lr.step()

        print(f'Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(vae.state_dict(), cfg.vae_ckpt)
            print(f'  → VAE saved (val_loss={val_loss:.4f})')

    print('VAE training complete.')


# train_vae(vae, train_dl, val_dl, cfg)

## Cell 14 — Diffusion Training Loop

In [ ]:
def train_diffusion(unet, mask_enc, vae, scheduler,
                    train_dl, val_dl, cfg):
    # Freeze VAE — only train U-Net + mask encoder
    vae.eval()
    for p in vae.parameters():
        p.requires_grad_(False)

    params      = list(unet.parameters()) + list(mask_enc.parameters())
    optimizer   = torch.optim.AdamW(params,
                                    lr=cfg.lr,
                                    weight_decay=cfg.weight_decay)

    # Warmup + cosine decay
    def lr_lambda(step):
        if step < cfg.warmup_steps:
            return step / cfg.warmup_steps
        progress = (step - cfg.warmup_steps) / \
                   max(1, cfg.diff_epochs * len(train_dl) - cfg.warmup_steps)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * progress)))

    lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    ema          = EMA(unet, cfg.ema_decay)
    scaler       = torch.cuda.amp.GradScaler()
    global_step  = 0
    best_val     = float('inf')

    for epoch in range(cfg.diff_epochs):
        unet.train()
        mask_enc.train()
        train_losses = []

        for batch in tqdm(train_dl,
                          desc=f'Diffusion Epoch {epoch+1}/{cfg.diff_epochs}'):
            ct   = batch['ct'].to(DEVICE)     # (B, 1, 512, 512)
            mask = batch['mask'].to(DEVICE)   # (B, 1, 512, 512)

            with torch.no_grad():
                # Encode CT → latent
                z, _, _ = vae.encode(ct)       # (B, 4, 64, 64)

            # Sample random timesteps
            t = torch.randint(0, cfg.T_train, (ct.shape[0],)).to(DEVICE)

            # Forward noising
            z_noisy, noise = scheduler.q_sample(z, t)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                # Encode mask
                mask_lat   = mask_enc(mask)    # (B, 4, 64, 64)
                # Predict noise
                pred_noise = unet(z_noisy, t, mask_lat)
                # Simple MSE loss on noise prediction
                loss       = F.mse_loss(pred_noise, noise)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()
            ema.update(unet)

            train_losses.append(loss.item())
            global_step += 1

        # ── Validation ──────────────────────────────────
        unet.eval()
        mask_enc.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_dl:
                ct   = batch['ct'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                z, _, _ = vae.encode(ct)
                t        = torch.randint(0, cfg.T_train, (ct.shape[0],)).to(DEVICE)
                z_noisy, noise = scheduler.q_sample(z, t)
                with torch.cuda.amp.autocast():
                    mask_lat   = mask_enc(mask)
                    pred_noise = unet(z_noisy, t, mask_lat)
                    loss       = F.mse_loss(pred_noise, noise)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        print(f'Epoch {epoch+1:3d} | '
              f'Train: {train_loss:.5f} | Val: {val_loss:.5f} | '
              f'LR: {lr_scheduler.get_last_lr()[0]:.2e}')

        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                'unet':      unet.state_dict(),
                'unet_ema':  ema.state_dict(),
                'mask_enc':  mask_enc.state_dict(),
                'epoch':     epoch,
                'val_loss':  val_loss,
            }, cfg.diff_ckpt)
            print(f'  → Diffusion model saved (val={val_loss:.5f})')

    print('Diffusion training complete.')


# After loading data and training VAE:
# train_diffusion(unet, mask_enc, vae, scheduler, train_dl, val_dl, cfg)

## Cell 15 — Inference: Generate Synthetic CT Slices

In [ ]:
@torch.no_grad()
def generate_synthetic_ct(unet, mask_enc, vae, scheduler, cfg,
                           mask_batch: torch.Tensor,
                           use_ema: bool = True,
                           ema_obj: EMA = None) -> torch.Tensor:
    """
    Generate synthetic CT slices conditioned on nodule masks.

    mask_batch: (B, 1, 512, 512) — nodule segmentation masks
    returns:    (B, 1, 512, 512) — synthetic CT slices in HU
    """
    model = ema_obj.ema_model if (use_ema and ema_obj) else unet
    model.eval()
    mask_enc.eval()
    vae.eval()

    mask_batch = mask_batch.to(DEVICE)
    B = mask_batch.shape[0]

    # DDIM sampling in latent space
    shape    = (B, cfg.latent_channels, cfg.latent_size, cfg.latent_size)
    z_clean  = scheduler.ddim_sample(model, mask_enc, mask_batch, shape, eta=0.0)

    # Decode latent → CT image
    ct_synth = vae.decode(z_clean)   # (B, 1, 512, 512) in [-1, 1]
    ct_synth = ct_synth.cpu().numpy()

    # Denormalize to HU
    ct_hu    = denormalize_ct(ct_synth, cfg.hu_min, cfg.hu_max)
    return ct_hu


def visualize_results(ct_real: np.ndarray,
                      ct_synth: np.ndarray,
                      mask: np.ndarray,
                      n: int = 4):
    """Plot real CT, mask, and synthetic CT side by side."""
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    for i in range(n):
        axes[i, 0].imshow(ct_real[i, 0],  cmap='gray', vmin=-1000, vmax=400)
        axes[i, 0].set_title('Real CT (HU)')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask[i, 0],     cmap='hot')
        axes[i, 1].set_title('Nodule Mask')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(ct_synth[i, 0], cmap='gray', vmin=-1000, vmax=400)
        axes[i, 2].set_title('Synthetic CT (HU)')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig(f'{cfg.output_dir}/synthetic_ct_samples.png', dpi=150)
    plt.show()


print('Inference functions defined.')

## Cell 16 — Evaluation Metrics

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr


def compute_mae(real: np.ndarray, synth: np.ndarray) -> float:
    """Mean Absolute Error in HU."""
    return np.mean(np.abs(real - synth))


def compute_psnr(real: np.ndarray, synth: np.ndarray,
                 data_range: float = 1400.0) -> float:
    """PSNR — data_range = hu_max - hu_min = 400-(-1000) = 1400."""
    return psnr(real, synth, data_range=data_range)


def compute_ssim(real: np.ndarray, synth: np.ndarray,
                 data_range: float = 1400.0) -> float:
    """SSIM over a batch of slices."""
    scores = []
    for i in range(real.shape[0]):
        s = ssim(real[i, 0], synth[i, 0], data_range=data_range)
        scores.append(s)
    return np.mean(scores)


def compute_hu_per_tissue(synth: np.ndarray) -> dict:
    """Mean HU per tissue class in synthetic CT."""
    return {
        'air':         synth[synth < -900].mean()             if (synth < -900).any()             else None,
        'lung':        synth[(synth>=-900)&(synth<-300)].mean()if ((synth>=-900)&(synth<-300)).any()else None,
        'soft_tissue': synth[(synth>=-300)&(synth<=100)].mean()if ((synth>=-300)&(synth<=100)).any()else None,
        'bone':        synth[synth > 300].mean()              if (synth > 300).any()              else None,
    }


def evaluate_batch(real_ct: np.ndarray, synth_ct: np.ndarray) -> dict:
    """Run all metrics on a batch."""
    metrics = {
        'MAE (HU)': compute_mae(real_ct, synth_ct),
        'PSNR':     compute_psnr(real_ct, synth_ct),
        'SSIM':     compute_ssim(real_ct, synth_ct),
        'HU per tissue': compute_hu_per_tissue(synth_ct),
    }
    for k, v in metrics.items():
        if k != 'HU per tissue':
            print(f'{k:15s}: {v:.4f}')
        else:
            print(f'{k}:')
            for tissue, hu in v.items():
                if hu is not None:
                    print(f'  {tissue:15s}: {hu:.1f} HU')
    return metrics


print('Evaluation metrics defined.')

## Cell 17 — Full Pipeline: Load Checkpoint & Run Inference

In [ ]:
def load_models(cfg: Config):
    """Load trained VAE, U-Net, and mask encoder from checkpoints."""
    # VAE
    vae = VAE(cfg).to(DEVICE)
    vae.load_state_dict(torch.load(cfg.vae_ckpt, map_location=DEVICE))
    vae.eval()
    print('VAE loaded.')

    # U-Net + mask encoder
    unet     = UNet(cfg).to(DEVICE)
    mask_enc = MaskEncoder(cfg.in_channels, cfg.latent_channels).to(DEVICE)
    ema_obj  = EMA(unet, cfg.ema_decay)

    ckpt = torch.load(cfg.diff_ckpt, map_location=DEVICE)
    unet.load_state_dict(ckpt['unet'])
    ema_obj.load_state_dict(ckpt['unet_ema'])
    mask_enc.load_state_dict(ckpt['mask_enc'])

    unet.eval()
    mask_enc.eval()
    print(f'Diffusion model loaded (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.5f})')

    return vae, unet, mask_enc, ema_obj


def run_full_inference(cfg: Config, test_dl: DataLoader, n_batches: int = 2):
    """Load models, generate synthetic CTs, evaluate and visualize."""
    vae, unet, mask_enc, ema_obj = load_models(cfg)
    scheduler = NoiseScheduler(cfg)

    all_real, all_synth, all_masks = [], [], []

    for i, batch in enumerate(test_dl):
        if i >= n_batches:
            break
        ct_real  = batch['ct'].numpy()    # (B, 1, 512, 512) normalized
        mask     = batch['mask']          # (B, 1, 512, 512)

        # Generate
        ct_synth = generate_synthetic_ct(
            unet, mask_enc, vae, scheduler, cfg,
            mask_batch=mask, use_ema=True, ema_obj=ema_obj
        )  # HU values

        # Denormalize real CT to HU for comparison
        ct_real_hu = denormalize_ct(ct_real, cfg.hu_min, cfg.hu_max)

        all_real.append(ct_real_hu)
        all_synth.append(ct_synth)
        all_masks.append(mask.numpy())

    all_real  = np.concatenate(all_real,  axis=0)
    all_synth = np.concatenate(all_synth, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)

    # Evaluate
    print('\n── Evaluation Metrics ─────────────────')
    evaluate_batch(all_real, all_synth)

    # Visualize
    visualize_results(all_real, all_synth, all_masks, n=4)


# run_full_inference(cfg, test_dl, n_batches=2)

## Cell 18 — Model Summary & Parameter Count


In [ ]:
def count_parameters(model: nn.Module, name: str):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{name:25s} | Total: {total:>12,} | Trainable: {trainable:>12,}')


print('─' * 70)
print(f'{"Component":25s} | {"Total params":>12s} | {"Trainable":>12s}')
print('─' * 70)

vae_temp      = VAE(cfg).to(DEVICE)
unet_temp     = UNet(cfg).to(DEVICE)
mask_enc_temp = MaskEncoder(cfg.in_channels, cfg.latent_channels).to(DEVICE)

count_parameters(vae_temp,      'VAE')
count_parameters(mask_enc_temp, 'Mask Encoder')
count_parameters(unet_temp,     'Diffusion U-Net')

total_all = sum(p.numel() for m in [vae_temp, mask_enc_temp, unet_temp]
                for p in m.parameters())
print('─' * 70)
print(f'{"TOTAL":25s} | {total_all:>12,}')
print('─' * 70)

del vae_temp, unet_temp, mask_enc_temp
torch.cuda.empty_cache()